In [ ]:
import sys
sys.path.append('../')
import os
os.environ["CUDA_VISIBLE_DEVICES"] = '0'

import torch
from tqdm import tqdm
from Network.SEW7BNet.SModels_BF import *
from spikingjelly.datasets.dvs128_gesture import DVS128Gesture

torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = True

In [ ]:
dataset_path = '/ssd/Datasets/DVS_Gesture/'
batch_size = 8
workers = 2
class_number = 11
timestep = 16

In [ ]:
train_dataset = DVS128Gesture(root=dataset_path, train=True, data_type='frame', frames_number=timestep, split_by='number')
val_dataset = DVS128Gesture(root=dataset_path, train=False, data_type='frame', frames_number=timestep, split_by='number')

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False,num_workers=workers, pin_memory=True)

In [ ]:
model = SEWResNet(alpha=1.0)
model_path = f'../models/dvsgesture_7bnet_t{timestep}.pth.tar'

print(f"=> loading checkpoint '{model_path}'")
checkpoint = torch.load(model_path)
model = torch.nn.DataParallel(model).cuda()
functional.set_step_mode(model, step_mode='m')
best_acc1 = checkpoint['best_acc1']
model.load_state_dict(checkpoint['state_dict'])
print(f"=> loaded checkpoint '{model_path}' (epoch {checkpoint['epoch']}) best_acc1: {best_acc1:.3f}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()) / 1000000}")

In [ ]:
Confusion_Matrix = torch.zeros((class_number, class_number))
model.eval()
with torch.no_grad():
    for img, label in tqdm(val_loader):
        img = img.cuda()
        label = label.cuda()
        out_fr = model(img)
        guess = out_fr.argmax(1)
        for j in range(len(label)):
            Confusion_Matrix[label[j],guess[j]] += 1
        functional.reset_net(model)
acc = Confusion_Matrix.diag()
acc = acc.sum()/Confusion_Matrix.sum()
print(f'Confusion_Matrix = \n{Confusion_Matrix}')
print(f'acc = {acc}')